# 01-lossless-structure — 아무것도 안 버리고 줄이기

압축이라고 하면 보통 "덜 중요한 걸 버린다" 를 떠올립니다. 이 랩은 반대입니다.
**아무것도 안 버리고** 토큰만 줄입니다.

한 줄로 보여드리면 이렇습니다.

```
압축 전   [{"id":"A-1","amount":1200},{"id":"A-2","amount":3400}]
압축 후   #TSV │ id  │ amount
          "A-1" │ 1200
          "A-2" │ 3400
```

`id` 와 `amount` 라는 글자가 레코드마다 반복되던 것을 헤더 한 줄로 올렸습니다.
값은 하나도 안 바뀌었으니 **되돌리면 원본과 똑같습니다.**

가능한 이유는 텍스트에 **내용이 아닌데도 토큰을 먹는 부분**이 있기 때문입니다.

| 무엇이 중복인가 | 어디에 |
|---|---|
| 레코드마다 반복되는 **키 이름** | JSON 배열, API 응답 |
| 줄마다 반복되는 **타임스탬프·서비스명** | 로그 |
| 사람이 보라고 넣은 **들여쓰기** | JSON, XML |
| 세로줄을 맞추려고 채운 **정렬 공백** | 마크다운 표, 설정 파일 |

산문에는 이런 중복이 없습니다. 그래서 같은 코드가 구조화 텍스트에서는
28% 줄이고 산문에서는 1% 줄입니다. **입력이 결과를 정합니다.**

### 이 노트북에서 하실 것

| 절 | 무엇을 |
|---|---|
| 2 | 변환 7종이 실제로 무엇을 바꾸는지 전후로 봅니다 |
| 3 | "아무것도 안 잃었다" 를 어떻게 확인하는지 봅니다 |
| 4 | 그 확인이 **진짜로 잡는지** 일부러 고장 내서 봅니다 |
| 5~7 | 설정 3개를 전부 돌려 비교합니다 |

## 1. kit 과 변환 모듈 불러오기

In [ ]:
import sys
from pathlib import Path

LAB = Path.cwd().resolve()
LABS = LAB.parents[0]                  # labs/<이 랩> -> labs
sys.path.insert(0, str(LABS))
sys.path.insert(0, str(LAB))           # 이 랩의 모듈(transforms, blocks 등)

from kit import VERSION, config as C, dataset, env, metrics, tokens as T
from kit.display import table, pct
from kit.runner import Run

# .env 는 labs/.env → 저장소 루트 .env → scripts/explore/.env 순으로 찾습니다.
env.load(verbose=True)

RUNS = LABS.parent / "runs"
print("kit", VERSION, "· 랩", LAB.name)

import json

import transforms as X
from compress import compress, verify_steps

counter = T.make_counter({}, "gpt-5.4")


def eye(text, width=44):
    """눈에 안 보이는 글자를 보이게 바꿉니다.

    줄바꿈은 ⏎, 표 구분자(\\x1f)는 ▏, 연달아 나오는 공백은 · 로 표시합니다.
    안 그러면 표 안에서 줄과 공백이 뭉개져서, 정작 무엇이 바뀌었는지
    알아볼 수가 없습니다. 특히 공백을 지우는 변환은 전후가 똑같아 보입니다.
    """
    import re
    t = text.replace("\x1f", " ▏ ").replace("\n", " ⏎ ")
    t = re.sub(r" {2,}", lambda m: "·" * min(len(m.group()), 10), t)
    return t if len(t) <= width else t[:width - 1] + "…"


print("kit 준비 완료 · 변환", len(X.REGISTRY), "종")

## 2. 변환 7종이 실제로 무엇을 하는지 한 번에 보기

설명보다 **결과를 보시는 편이 빠릅니다.** 각 변환에 딱 맞는 짧은 입력을
하나씩 주고, 전후를 나란히 놓았습니다.

`⏎` 는 줄바꿈, `▏` 는 표 구분자입니다. 원래 눈에 안 보이는 글자라서
표시해 두었습니다.

In [ ]:
DEMO = {
    "json_to_table":    '[{"id":"A-1","amt":1200},{"id":"A-2","amt":3400}]',
    "json_compact":     '{\n  "host": "db-01",\n  "port": 5432\n}',
    "log_dedup":        ("2026-03-03T14:20:11 INFO  handler=refund ok\n"
                         "2026-03-03T14:20:11 INFO  handler=cancel ok\n"
                         "2026-03-03T14:20:11 ERROR handler=refund fail"),
    "xml_compact":      '<cfg>\n    <db host="a" port="1"/>\n</cfg>',
    "md_table_compact": ("| 요금제  | 가격  |\n|--------|------|\n"
                         "| Free   | 0    |\n| Pro    | 49000|"),
    "kv_compact":       "host    :   db-01\nport    :   5432\nretry   :   3",
    "ws_collapse":      "a      b       c       d",
}

rows = []
for name, src in DEMO.items():
    out, meta = compress(src, pipeline=[name])
    ok, why, checked = verify_steps(meta["_steps"])
    rows.append([
        name,
        eye(src),
        eye(out),
        f"{len(src)} → {len(out)}",
        f"-{1 - len(out) / len(src):.0%}",
        "검증됨" if checked else "검증 불가",
    ])

table(
    ["변환", "압축 전", "압축 후", "글자", "절감", "확인"],
    rows,
    align=["left", "left", "left", "right", "right", "left"],
    title="변환 7종 · 같은 입력을 주면 이렇게 바뀝니다",
    note="맨 아래 ws_collapse 만 '검증 불가' 입니다. 이유는 다음 절에서 설명드립니다.",
)

## 3. "아무것도 안 잃었다" 를 어떻게 확인하나요

줄었다는 건 글자 수만 세면 바로 보입니다. 어려운 건 **아무것도 안 잃었다**
쪽입니다. 위 표의 `json_to_table` 하나를 붙잡고 실제로 확인해 보겠습니다.

In [ ]:
before = json.dumps([{"id": "A-1", "amount": 1200, "status": "paid"},
                     {"id": "A-2", "amount": 3400, "status": "refunded"}],
                    ensure_ascii=False, indent=2)
after, meta = compress(before, pipeline=["json_to_table"])

print("━━━ 압축 전 ━━━")
print(before)
print(f"\n{len(before)}자 · {counter(before)} 토큰\n")

print("━━━ 압축 후 ━━━")
print(after.replace("\x1f", " ▏ "))
print(f"\n{len(after)}자 · {counter(after)} 토큰")
print(f"\n키 이름 id·amount·status 가 레코드마다 반복되던 것이 헤더 한 줄로 갔습니다.")

줄어든 건 확인했습니다. 그럼 **잃은 건 없을까요?**

압축 결과를 다시 펴서 원본과 맞춰봅니다. 두 개가 같으면 잃은 게 없다는 뜻입니다.

In [ ]:
restored = X.json_to_table_restore(after)      # 압축본을 다시 폅니다
original = json.loads(before)                  # 원본을 파싱합니다

print("원본을 파싱  :", original)
print("압축본을 되폄:", restored)
print()
print("두 값이 같은가:", restored == original, "← 같으면 잃은 것이 없습니다")

ok, why, checked = verify_steps(meta["_steps"])
print(f"\n랩이 매 케이스마다 하는 검사: {ok} · {why}")

### 확인하는 방법이 두 가지인 이유

방금 본 것은 **다시 펴서 맞춰보기**였습니다. 그런데 이 방법이 안 통하는
변환도 있습니다.

```
json_compact:   {                      →   {"host":"db-01","port":5432}
                  "host": "db-01",
                  "port": 5432
                }

되돌려 보면?    {"host": "db-01", "port": 5432}
                ↑ 들여쓰기가 몇 칸이었는지 복원할 수 없습니다
```

**글자 단위로는 원본과 다릅니다.** 그렇다고 정보를 잃은 걸까요? 아닙니다.
JSON 에서 들여쓰기는 내용이 아니기 때문입니다. 그래서 이럴 때는 **양쪽을
파싱해서 객체끼리** 맞춰봅니다.

| 확인 방법 | 무엇과 무엇을 비교하나요 | 쓰는 변환 |
|---|---|---|
| **되돌리기** | 되돌린 글자 ↔ 원본 글자 | `log_dedup` |
| **정규형** | 파싱한 값 ↔ 파싱한 값 | `json_*`, `xml_*`, `md_table_*`, `kv_*` |
| (없음) | 비교할 방법이 없습니다 | `ws_collapse` |

`ws_collapse`(공백 접기)만 두 방법 다 안 됩니다. `a      b` 를 `a b` 로
바꾸면 원래 공백이 몇 칸이었는지도, 무엇과 비교해야 할지도 알 수 없습니다.
그래서 이 랩은 **`ws_collapse` 를 무손실이 아니라 손실로 분류**하고,
쓴 횟수를 결과에 남깁니다.

In [ ]:
rows = []
for name in ["log_dedup", "json_compact", "ws_collapse"]:
    t = X.REGISTRY[name]
    how = "되돌리기" if t.restore else ("정규형 비교" if t.canon else "없음")
    src = DEMO[name]
    out, meta = compress(src, pipeline=[name])
    ok, why, checked = verify_steps(meta["_steps"])
    rows.append([name, how, "예" if checked else "아니요", why])

table(
    ["변환", "확인 방법", "검증했나요", "결과"],
    rows,
    align=["left", "left", "center", "left"],
    title="세 가지 경우를 나란히",
    note="검증 못 한 변환을 쓰면 그 실행 결과는 '무손실' 이라고 부를 수 없습니다.",
)

## 4. 검사가 진짜 잡는지 확인하기

앞 절에서 "검증했습니다" 라는 결과를 봤습니다. 그런데 그 검사가 정말로
일하고 있는 걸까요, 아니면 그냥 항상 통과만 하는 걸까요?

**일부러 고장 낸 입력을 넣어봐야 알 수 있습니다.** 값을 몰래 지우는 변환을
심어서 검사가 걸러내는지 봅니다.

In [ ]:
def sneaky(text):
    """status 필드를 몰래 버리면서 무손실인 척하는 변환입니다."""
    o = json.loads(text)
    for r in o:
        r.pop("status", None)
    return True, json.dumps(o, ensure_ascii=False, separators=(",", ":")), {}


X.REGISTRY["sneaky"] = X.Transform("sneaky", sneaky, canon=X.json_canon)

rows = []
for name, label in [("json_to_table", "정상 변환"), ("sneaky", "몰래 삭제")]:
    out, meta = compress(before, pipeline=[name])
    ok, why, _ = verify_steps(meta["_steps"])
    rows.append([label, eye(out, 40), "통과" if ok else "걸림", why])

table(
    ["무엇을 넣었나", "결과물", "검사", "판정"],
    rows,
    align=["left", "left", "center", "left"],
    title="정상 변환 vs 값을 몰래 버리는 변환",
    note="아래쪽이 '걸림' 으로 나와야 검사가 일하고 있는 것입니다.",
)

out, meta = compress(before, pipeline=["sneaky"])
print("몰래 삭제된 결과 :", out)
print("status 필드가 없어졌는데도 JSON 으로는 멀쩡해 보입니다.")
print("정규형 비교가 아니었다면 그냥 통과했을 것입니다.")

del X.REGISTRY["sneaky"]

## 5. 이 랩의 모든 조건 돌려보기

**조건 1개 = 파일 1개**입니다. `configs/` 를 훑으면 이 랩이 답할 수 있는
질문이 전부 나옵니다. 설정을 새로 추가해도 이 셀은 고칠 필요가 없습니다.

각 조건은 `runs/01-lossless-structure/<설정이름>/<시각>/` 에 따로 기록됩니다. 나중에
"그때 무엇을 돌렸나" 를 설정 이름만 보고 알 수 있게 하려는 것입니다.

| 설정 | 입력 | 무엇을 보려고 |
|---|---|---|
| `structure` | 구조화 12건 | 이 랩의 기본 조건 |
| `prose` | 산문 12건 | **대조군** — 같은 코드가 산문에서 무엇을 하나 |
| `structure-lossy-ws` | 구조화 12건 | 공백까지 접으면 얼마를 더 얻고 무엇을 잃나 |

In [ ]:
def run_config(path):
    cfg = C.load(path)
    cases = dataset.load(cfg.dataset["path"], limit=cfg.dataset.get("limit"))
    counter = T.make_counter(cfg.tokenizer, cfg.model)

    run = Run(cfg, RUNS)
    broken, applied_count, untouched = [], {}, 0
    n_checked = n_unchecked = 0

    for c in cases:
        after, extra = compress(c.text, **cfg.params)
        steps = extra.pop("_steps")
        ok, why, checked = verify_steps(steps)
        extra["verified"] = why
        n_checked += checked
        n_unchecked += len(steps) - checked
        if not ok:
            broken.append((c.id, why))
        for n in extra["applied"]:
            applied_count[n] = applied_count.get(n, 0) + 1
        if not extra["applied"]:
            untouched += 1
        run.add(metrics.per_case(c.id, c.kind, c.text, after, c.must_include,
                                 counter, extra),
                before=c.text, after=after)

    m = metrics.aggregate(run.records, counter)
    m.update({"dataset_name": Path(cfg.dataset["path"]).name,
              "applied_count": applied_count, "untouched": untouched,
              "steps_verified": n_checked, "steps_unverified": n_unchecked,
              "broken": broken})
    return cfg, m, run.finish(m, [f"적용 횟수 {applied_count or '없음'}"])


results = []
for p in sorted(Path("configs").glob("*.yaml")):
    cfg, m, out = run_config(p)
    results.append((cfg.name, m, out))
    flag = " ← 검증 불가 포함" if m["steps_unverified"] else ""
    print(f'{cfg.name:22s} 절감 {m["saved"]:6.1%} · '
          f'검증 {m["steps_verified"]:2d}단계 · 손 안 댐 {m["untouched"]:2d}건{flag}')

## 6. 조건 비교

같은 코드에 조건만 바꿔 돌린 결과입니다. **숫자 하나가 아니라 표를 보세요.**
어떤 조건에서 무엇을 얻고 무엇을 잃는지가 이 랩의 결론입니다.

**여기서 읽어야 할 것 두 가지입니다.**

1. `structure` 와 `prose` 는 **코드가 한 글자도 안 다릅니다.** 입력만 다릅니다.
2. `structure-lossy-ws` 는 절감이 조금 늘지만 **검증 불가 단계**가 생깁니다.
   그만큼 "무손실" 이라는 말을 쓸 수 없게 됩니다.

In [ ]:
table(
    ["설정", "코퍼스", "절감", "최저 보존율", "검증", "검증 불가", "손 안 댐"],
    [[n, m["dataset_name"], pct(m["saved"]), pct(m.get("survival_worst")),
      f'{m["steps_verified"]}단계',
      f'{m["steps_unverified"]}단계' if m["steps_unverified"] else "없음",
      f'{m["untouched"]}건']
     for n, m, _ in results],
    align=["left", "left", "right", "right", "right", "right", "right"],
    title="조건 비교",
    note="검증 불가 단계가 하나라도 있으면 그 조건의 결과는 무손실이 아닙니다.",
)

for n, m, _ in results:
    if m["broken"]:
        print(f"✗ {n}: 정보 손실 {len(m['broken'])}건 — {m['broken'][:2]}")

by = {n: m for n, m, _ in results}
if "structure" in by and "prose" in by:
    print(f'같은 코드, 다른 입력: 구조화 {by["structure"]["saved"]:.1%} '
          f'vs 산문 {by["prose"]["saved"]:.1%}')
if "structure" in by and "structure-lossy-ws" in by:
    d = by["structure-lossy-ws"]["saved"] - by["structure"]["saved"]
    print(f'공백까지 접어서 더 얻은 것: {d:.1%}p — '
          f'그 대가로 무손실 보장을 잃습니다.')

## 7. 유형별 — 어디서 이득이 나나

무손실의 이득은 **중복의 양에 비례**합니다. 그래서 유형마다 크게 다릅니다.

In [ ]:
m = by.get("structure") or results[0][1]

table(
    ["유형", "건수", "절감", "왜"],
    [[k, v["n"], pct(v["saved"]), {
        "json-array": "레코드가 많을수록 키 반복이 커집니다",
        "json-nested": "들여쓰기가 통째로 사라집니다",
        "kv-space": "콜론을 맞추려고 채운 공백이 전부 장식입니다",
        "md-table": "정렬 패딩과 구분선이 사라집니다",
        "log-repeat": "접두사는 길지만 줄 수가 적으면 이득도 적습니다",
        "xml": "태그 이름 자체는 못 줄입니다",
     }.get(k, "")]
     for k, v in sorted(m["by_kind"].items(), key=lambda x: -x[1]["saved"])],
    align=["left", "right", "right", "left"],
    title="유형별 절감 (structure 조건)",
)

print("적용 횟수:", m["applied_count"])

## 정리

- **무손실은 표현의 중복을 먹습니다** — 내용이 아니라 포맷을 줄입니다
- **입력이 결과를 정합니다** — 같은 코드가 28% 도 되고 1% 도 됩니다
- **검증할 수 없으면 무손실이 아닙니다** — 되돌리기나 정규형 비교 중 하나는 있어야 합니다
- **공백 접기는 대개 손해입니다** — 조금 더 얻고 보장을 잃습니다

무손실은 여기까지가 천장입니다. 더 줄이려면 **무언가는 버려야** 합니다.

### 다음 랩

[`02-handle-ref`](../02-handle-ref/run.ipynb) — 버리는 대신 밖에 두고
필요할 때만 꺼냅니다.